# E8 Error Analysis
Full Spider dev (1034 examples) — value linking + column linking + tool-augmented refiner

In [1]:
import json
import pandas as pd
from collections import Counter
from pathlib import Path

RESULTS_PATH = Path("../outputs/ablation_cheap_gen_more_candidates_20260410_013329_20260410_013331.json")

with open(RESULTS_PATH) as f:
    data = json.load(f)

preds = data["predictions"]
df = pd.DataFrame(preds)
print(f"Total examples: {len(df)}")
print(f"EX: {df['execution_match'].mean():.2%}")
print(f"EM: {df['exact_match'].mean():.2%}")
print(f"Hard errors: {df['error_message'].notna().sum()}")
print(f"Cost: ${df['total_cost_usd'].sum():.2f}")

Total examples: 1034
EX: 74.95%
EM: 35.78%
Hard errors: 37
Cost: $30.60


## 1. Error breakdown by db_id

In [2]:
failures = df[~df["execution_match"]].copy()
print(f"Total failures: {len(failures)}\n")

db_errors = failures.groupby("db_id").agg(
    failures=("execution_match", "count"),
    hard_errors=("error_message", lambda x: x.notna().sum()),
).sort_values("failures", ascending=False)

db_total = df.groupby("db_id").size().rename("total")
db_errors = db_errors.join(db_total)
db_errors["fail_rate"] = (db_errors["failures"] / db_errors["total"] * 100).round(1)
db_errors = db_errors[["total", "failures", "hard_errors", "fail_rate"]]
print(db_errors.to_string())

Total failures: 259

                              total  failures  hard_errors  fail_rate
db_id                                                                
car_1                            92        51           19       55.4
student_transcripts_tracking     78        43            5       55.1
world_1                         120        25            3       20.8
dog_kennels                      82        23            2       28.0
flight_2                         80        22            0       27.5
employee_hire_evaluation         38        17            3       44.7
concert_singer                   45        12            3       26.7
cre_Doc_Template_Mgt             84        10            0       11.9
tvshow                           62        10            0       16.1
network_1                        56         9            0       16.1
wta_1                            62         9            2       14.5
orchestra                        40         8            0       20.0

## 2. Hard error categories

In [3]:
hard_errors = df[df["error_message"].notna()].copy()

def categorize_error(msg):
    if not msg:
        return "none"
    msg = str(msg)
    if "benchmark_timeout" in msg:
        return "timeout"
    if "schema_validation" in msg:
        if "unknown table" in msg:
            return "schema: unknown table"
        if "unknown column" in msg:
            return "schema: unknown column"
        return "schema: other"
    if "no such column" in msg:
        return "sqlite: no such column"
    if "no such table" in msg:
        return "sqlite: no such table"
    if "Could not decode" in msg or "UTF" in msg:
        return "encoding"
    if "OperationalError" in msg:
        return "sqlite: other"
    return "other"

hard_errors["category"] = hard_errors["error_message"].apply(categorize_error)

cat_counts = hard_errors["category"].value_counts()
print("Error categories:")
for cat, cnt in cat_counts.items():
    print(f"  {cat}: {cnt}")
print(f"\nTotal hard errors: {len(hard_errors)}")

Error categories:
  schema: unknown table: 11
  schema: unknown column: 9
  sqlite: no such column: 9
  timeout: 6
  encoding: 2

Total hard errors: 37


In [4]:
print("Hard errors by db_id + category:\n")
pivot = hard_errors.groupby(["db_id", "category"]).size().unstack(fill_value=0)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=False)
print(pivot.to_string())

Hard errors by db_id + category:

category                      encoding  schema: unknown column  schema: unknown table  sqlite: no such column  timeout  total
db_id                                                                                                                        
car_1                                0                       5                      5                       6        3     19
student_transcripts_tracking         0                       0                      3                       2        0      5
concert_singer                       0                       2                      0                       0        1      3
employee_hire_evaluation             0                       0                      0                       1        2      3
world_1                              0                       0                      3                       0        0      3
dog_kennels                          0                       2                      

## 3. Soft failures analysis (EX=False, no hard error)

In [5]:
soft_failures = failures[failures["error_message"].isna()].copy()
print(f"Soft failures (wrong result, no crash): {len(soft_failures)}")
print(f"\nBy db_id (top 15):")
soft_by_db = soft_failures["db_id"].value_counts().head(15)
for db, cnt in soft_by_db.items():
    total = len(df[df["db_id"] == db])
    print(f"  {db}: {cnt}/{total} ({cnt/total:.0%})")

Soft failures (wrong result, no crash): 222

By db_id (top 15):
  student_transcripts_tracking: 38/78 (49%)
  car_1: 32/92 (35%)
  flight_2: 22/80 (28%)
  world_1: 22/120 (18%)
  dog_kennels: 21/82 (26%)
  employee_hire_evaluation: 14/38 (37%)
  cre_Doc_Template_Mgt: 10/84 (12%)
  tvshow: 10/62 (16%)
  concert_singer: 9/45 (20%)
  network_1: 9/56 (16%)
  orchestra: 8/40 (20%)
  wta_1: 7/62 (11%)
  poker_player: 5/40 (12%)
  pets_1: 3/42 (7%)
  museum_visit: 3/18 (17%)


## 4. EX accuracy by db_id

In [6]:
db_acc = df.groupby("db_id").agg(
    total=("execution_match", "count"),
    correct=("execution_match", "sum"),
)
db_acc["ex"] = (db_acc["correct"] / db_acc["total"] * 100).round(1)
db_acc = db_acc.sort_values("ex")

print("Worst db_id by EX (bottom 15):")
print(db_acc.head(15).to_string())
print(f"\nBest db_id by EX (top 10):")
print(db_acc.tail(10).to_string())

Worst db_id by EX (bottom 15):
                              total  correct    ex
db_id                                             
car_1                            92       41  44.6
student_transcripts_tracking     78       35  44.9
real_estate_properties            4        2  50.0
employee_hire_evaluation         38       21  55.3
dog_kennels                      82       59  72.0
flight_2                         80       58  72.5
concert_singer                   45       33  73.3
world_1                         120       95  79.2
orchestra                        40       32  80.0
battle_death                     16       13  81.2
museum_visit                     18       15  83.3
tvshow                           62       52  83.9
network_1                        56       47  83.9
wta_1                            62       53  85.5
voter_1                          15       13  86.7

Best db_id by EX (top 10):
                      total  correct     ex
db_id                         

## 5. Predicted vs Gold SQL comparison for soft failures

In [7]:
import re

def classify_mismatch(pred, gold):
    """Heuristic labels for near-miss failures."""
    pred_u = pred.upper().strip().rstrip(";")
    gold_u = gold.upper().strip().rstrip(";")
    
    labels = []
    
    # projection width
    pred_sel = re.search(r"SELECT\s+(.*?)\s+FROM", pred_u, re.DOTALL)
    gold_sel = re.search(r"SELECT\s+(.*?)\s+FROM", gold_u, re.DOTALL)
    if pred_sel and gold_sel:
        pred_cols = [c.strip() for c in pred_sel.group(1).split(",")]
        gold_cols = [c.strip() for c in gold_sel.group(1).split(",")]
        if len(pred_cols) != len(gold_cols):
            labels.append("projection-width")
        elif pred_cols != gold_cols and set(pred_cols) == set(gold_cols):
            labels.append("projection-order")
    
    # unnecessary join
    pred_joins = len(re.findall(r"\bJOIN\b", pred_u))
    gold_joins = len(re.findall(r"\bJOIN\b", gold_u))
    if pred_joins > gold_joins:
        labels.append("extra-join")
    elif pred_joins < gold_joins:
        labels.append("missing-join")
    
    # aggregation shape
    for agg in ["COUNT", "SUM", "AVG", "MIN", "MAX"]:
        in_pred = agg in pred_u
        in_gold = agg in gold_u
        if in_pred != in_gold:
            labels.append("aggregation-shape")
            break
    
    # DISTINCT mismatch
    if ("DISTINCT" in pred_u) != ("DISTINCT" in gold_u):
        labels.append("distinct-mismatch")
    
    # ORDER BY
    if ("ORDER BY" in pred_u) != ("ORDER BY" in gold_u):
        labels.append("ordering-mismatch")
    
    # LIMIT
    if ("LIMIT" in pred_u) != ("LIMIT" in gold_u):
        labels.append("limit-mismatch")
    
    # WHERE clause presence
    if ("WHERE" in pred_u) != ("WHERE" in gold_u):
        labels.append("filter-mismatch")
    
    # GROUP BY
    if ("GROUP BY" in pred_u) != ("GROUP BY" in gold_u):
        labels.append("grouping-mismatch")
    
    if not labels:
        labels.append("subtle")
    
    return labels

all_labels = []
for _, row in soft_failures.iterrows():
    labels = classify_mismatch(row["predicted_sql"], row["gold_sql"])
    all_labels.extend(labels)

label_counts = Counter(all_labels).most_common()
print(f"Mismatch type distribution ({len(soft_failures)} soft failures):\n")
for label, cnt in label_counts:
    print(f"  {label}: {cnt} ({cnt/len(soft_failures):.0%})")

Mismatch type distribution (222 soft failures):

  subtle: 83 (37%)
  missing-join: 57 (26%)
  aggregation-shape: 38 (17%)
  projection-width: 34 (15%)
  filter-mismatch: 32 (14%)
  distinct-mismatch: 31 (14%)
  extra-join: 25 (11%)
  grouping-mismatch: 16 (7%)
  ordering-mismatch: 12 (5%)
  limit-mismatch: 11 (5%)


## 6. Sample soft failures by category

In [8]:
soft_failures["mismatch_labels"] = soft_failures.apply(
    lambda r: classify_mismatch(r["predicted_sql"], r["gold_sql"]), axis=1
)

for label in ["extra-join", "projection-width", "aggregation-shape", "distinct-mismatch", "subtle"]:
    subset = soft_failures[soft_failures["mismatch_labels"].apply(lambda x: label in x)]
    if subset.empty:
        continue
    print(f"\n{'='*80}")
    print(f"Category: {label} ({len(subset)} examples)")
    print(f"{'='*80}")
    for _, row in subset.head(3).iterrows():
        print(f"\n[{row['db_id']}] {row['question']}")
        print(f"  PRED: {row['predicted_sql'][:200]}")
        print(f"  GOLD: {row['gold_sql'][:200]}")


Category: extra-join (25 examples)

[concert_singer] How many concerts are there in year 2014 or 2015?
  PRED: SELECT COUNT(DISTINCT concert_ID) FROM singer_in_concert JOIN singer ON singer_in_concert.Singer_ID = singer.Singer_ID WHERE singer.Song_release_year IN ('2014', '2015');
  GOLD: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015

[car_1] How many car models are produced in the usa?
  PRED: SELECT COUNT(DISTINCT T1.Model) FROM car_names AS T1 JOIN model_list AS T2 ON T1.Model = T2.Model JOIN car_makers AS T3 ON T2.Maker = T3.Id JOIN countries AS T4 ON T3.Country = T4.CountryId WHERE T4.C
  GOLD: SELECT count(*) FROM MODEL_LIST AS T1 JOIN CAR_MAKERS AS T2 ON T1.Maker  =  T2.Id JOIN COUNTRIES AS T3 ON T2.Country  =  T3.CountryId WHERE T3.CountryName  =  'usa';

[employee_hire_evaluation] Find the number of shops in each location.
  PRED: SELECT T1.City , COUNT(DISTINCT T2.Shop_ID) FROM employee AS T1 JOIN hiring AS T2 ON T1.Employee_ID = T2.Employee_ID GROUP BY T

## 7. Cost distribution

In [9]:
print("Cost per example (USD):")
print(df["total_cost_usd"].describe().to_string())
print(f"\nMedian: ${df['total_cost_usd'].median():.4f}")
print(f"P95:    ${df['total_cost_usd'].quantile(0.95):.4f}")
print(f"P99:    ${df['total_cost_usd'].quantile(0.99):.4f}")
print(f"Max:    ${df['total_cost_usd'].max():.4f}")

print(f"\nCost by correctness:")
print(f"  Correct:   ${df[df['execution_match']]['total_cost_usd'].mean():.4f}/example")
print(f"  Wrong:     ${df[~df['execution_match']]['total_cost_usd'].mean():.4f}/example")

Cost per example (USD):
count    1034.000000
mean        0.029591
std         0.018049
min         0.000000
25%         0.019565
50%         0.025532
75%         0.033371
max         0.203650

Median: $0.0255
P95:    $0.0564
P99:    $0.1123
Max:    $0.2036

Cost by correctness:
  Correct:   $0.0272/example
  Wrong:     $0.0369/example


## 8. Value/column linking coverage

In [10]:
def count_hints(warnings_list, prefix):
    if not warnings_list:
        return 0
    for w in warnings_list:
        if prefix in str(w):
            import re
            m = re.search(r"(\d+)", str(w).split(prefix)[-1])
            if m:
                return int(m.group(1))
    return 0

df["n_value_hints"] = df["warnings"].apply(lambda w: count_hints(w, "value hint"))
df["n_column_hints"] = df["warnings"].apply(lambda w: count_hints(w, "column hint"))
df["has_value_hints"] = df["n_value_hints"] > 0
df["has_column_hints"] = df["n_column_hints"] > 0

print("Linking coverage:")
print(f"  Value hints present:  {df['has_value_hints'].sum()}/{len(df)} ({df['has_value_hints'].mean():.0%})")
print(f"  Column hints present: {df['has_column_hints'].sum()}/{len(df)} ({df['has_column_hints'].mean():.0%})")
print(f"  Either:               {(df['has_value_hints'] | df['has_column_hints']).sum()}/{len(df)}")

print(f"\nEX by linking presence:")
for label, mask in [
    ("with value hints", df["has_value_hints"]),
    ("without value hints", ~df["has_value_hints"]),
    ("with column hints", df["has_column_hints"]),
    ("without column hints", ~df["has_column_hints"]),
]:
    subset = df[mask]
    print(f"  {label}: EX={subset['execution_match'].mean():.2%} (n={len(subset)})")

Linking coverage:
  Value hints present:  0/1034 (0%)
  Column hints present: 0/1034 (0%)
  Either:               0/1034

EX by linking presence:
  with value hints: EX=nan% (n=0)
  without value hints: EX=74.95% (n=1034)
  with column hints: EX=nan% (n=0)
  without column hints: EX=74.95% (n=1034)


## 9. Summary

In [11]:
print("="*60)
print("E8 ERROR ANALYSIS SUMMARY")
print("="*60)
print(f"\nTotal: {len(df)} examples")
print(f"EX: {df['execution_match'].mean():.2%} | EM: {df['exact_match'].mean():.2%}")
print(f"\nFailures: {len(failures)} ({len(failures)/len(df):.1%})")
print(f"  Hard errors: {len(hard_errors)}")
print(f"  Soft failures: {len(soft_failures)}")
print(f"\nTop failing DBs:")
for db, row in db_errors.head(5).iterrows():
    print(f"  {db}: {int(row['failures'])} failures / {int(row['total'])} total ({row['fail_rate']}%)")
print(f"\nTop error categories:")
for cat, cnt in cat_counts.head(5).items():
    print(f"  {cat}: {cnt}")
print(f"\nTop mismatch patterns:")
for label, cnt in label_counts[:5]:
    print(f"  {label}: {cnt}")

E8 ERROR ANALYSIS SUMMARY

Total: 1034 examples
EX: 74.95% | EM: 35.78%

Failures: 259 (25.0%)
  Hard errors: 37
  Soft failures: 222

Top failing DBs:
  car_1: 51 failures / 92 total (55.4%)
  student_transcripts_tracking: 43 failures / 78 total (55.1%)
  world_1: 25 failures / 120 total (20.8%)
  dog_kennels: 23 failures / 82 total (28.0%)
  flight_2: 22 failures / 80 total (27.5%)

Top error categories:
  schema: unknown table: 11
  schema: unknown column: 9
  sqlite: no such column: 9
  timeout: 6
  encoding: 2

Top mismatch patterns:
  subtle: 83
  missing-join: 57
  aggregation-shape: 38
  projection-width: 34
  filter-mismatch: 32
